In [2]:
from Training_utils import train_loader, train
import torch_geometric as tg
import torch
from SUPG_prediction_models import *

from Training_utils import *


tset = graph_dataset(f"data/training_set_edge_attr/input_values")


class gat(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.model = tg.nn.models.GAT(
            in_channels=10, 
            hidden_channels=5, 
            num_layers=10, 
            out_channels=1, 
            v2=True, 
            #dropout=0., 
            act=torch.relu, 
            #norm=torch_geometric.nn.norm.LayerNorm(1),
            add_self_loops=False,
            edge_dim=4,
            residual=False
        )

    def forward(self, data) -> torch.Tensor:
        x, edge_index, edge_attr, upper = data.x, data.edge_index, data.edge_attr, data.upper
        h = self.model(
            x=x,
            edge_index=edge_index,
            edge_attr=edge_attr
        )
        return upper*torch.sigmoid(h)
    
batch_size = 15
loader = train_loader(batch_size=batch_size, set=tset)
model=gat()


optimizer = torch.optim.Adam(model.parameters(), lr=0.001)


scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer=optimizer, factor=0.8, patience=50)
curr_loss = 10



In [6]:
batch_size = 50
loader = train_loader(batch_size=batch_size, set=tset)

In [4]:

for i in range(5000):
    loss = train(model=model, loader=loader, optimizer=optimizer, device='cpu')
    if curr_loss > loss:    
        print(f"iteration {i}: new loss: {loss}")
        curr_loss = loss
        torch.save({'model_state': model.state_dict(), 'optimizer_state': optimizer.state_dict(), 'loss': loss}, "data/models/GATv2_supervised_edge_attr.pth")
        #scheduler.step(loss)
    else:
        print(f"iteration {i}: {loss}")
        scheduler.step(loss)

        


iteration 0: new loss: 0.8839712646348
iteration 1: new loss: 0.8795831337351543
iteration 2: new loss: 0.873723933290669
iteration 3: new loss: 0.8608887155582621
iteration 4: new loss: 0.8452933274728077
iteration 5: new loss: 0.8313675140525447
iteration 6: new loss: 0.8229467230432254
iteration 7: new loss: 0.8219127189559831
iteration 8: new loss: 0.8198914294744927
iteration 9: new loss: 0.8193197596119717
iteration 10: new loss: 0.8192887583605956
iteration 11: new loss: 0.8183621376764759
iteration 12: 0.819649434940154
iteration 13: 0.8218132755408244
iteration 14: 0.824116139639624
iteration 15: new loss: 0.8173606745397344
iteration 16: new loss: 0.8156341268239141
iteration 17: new loss: 0.814909482962817
iteration 18: new loss: 0.8141150223372361
iteration 19: 0.8279307666029623
iteration 20: new loss: 0.8137839849785616
iteration 21: new loss: 0.8087577511793246
iteration 22: new loss: 0.8042718150235569
iteration 23: new loss: 0.8022129496123604
iteration 24: new loss: 0

KeyboardInterrupt: 